# Model C — Question-Flow / Dialogue Policy

Selects the single best next question to ask the patient based on what is known and what is missing.
No training data required — uses Gemini with clinical coverage rules.

One question per turn. No diagnosis. No free-form chat.

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
import re

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    generation_config=genai.types.GenerationConfig(temperature=0.2, max_output_tokens=80),
    safety_settings=[
        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_ONLY_HIGH"},
    ],
)

In [ ]:
@dataclass
class ConversationContext:
    questions_asked: List[str] = field(default_factory=list)
    patient_answers: List[str] = field(default_factory=list)

    @property
    def stage(self) -> str:
        n = len(self.questions_asked)
        if n <= 2:   return "early"
        if n <= 6:   return "mid"
        return "escalation"

    def add_turn(self, question: str, answer: str):
        self.questions_asked.append(question)
        self.patient_answers.append(answer)

In [ ]:
COVERAGE_CHECKLIST = [
    "severity or intensity of the main symptom",
    "when the symptom started or how long it has been present",
    "whether the symptom is getting better, worse, or staying the same",
    "any other symptoms alongside the main one",
    "whether the symptom affects the patient's daily activities",
    "any relevant medical history or known conditions",
]

RED_FLAG_FOLLOWUPS = [
    "Is the patient currently able to breathe comfortably?",
    "Has the patient lost consciousness or felt faint?",
    "Is there any unusual bleeding?",
    "Is the patient able to move all limbs normally?",
]

SYSTEM_PROMPT = """\
You are a clinical question-selection assistant in a hospital pre-consultation system.
Output ONE question only. Nothing else.

Rules:
- One question per response. No exceptions.
- No diagnosis, no medical advice, no interpretation.
- Do not repeat a question already asked.
- Keep it short and natural to say out loud.
- Output the question text only — no labels, no prefix.
"""

In [ ]:
def build_prompt(state: dict, context: ConversationContext) -> str:
    known = [f"{k}: {v}" for k, v in state.items() if v]

    history = "\n".join(
        f"  Q: {q}\n  A: {a}"
        for q, a in zip(context.questions_asked, context.patient_answers)
    ) or "  (none)"

    red_flag_block = ""
    if state.get("red_flags_present"):
        followups = "\n".join(f"- {q}" for q in RED_FLAG_FOLLOWUPS)
        red_flag_block = f"\nRED FLAG ACTIVE. Prioritise these:\n{followups}\n"

    coverage = "\n".join(f"- {c}" for c in COVERAGE_CHECKLIST)

    return f"""\
Patient state:
{chr(10).join(f'- {k}' for k in known) or '- (unknown)'}

Conversation so far:
{history}

Stage: {context.stage}
{red_flag_block}
Coverage checklist (must be addressed eventually):
{coverage}

Output the single best next question."""

In [ ]:
def select_next_question(state: dict, context: ConversationContext) -> str:
    """
    Select the next question to ask the patient.

    Args:
        state   : Dict of known patient fields (from Model B extraction_dict).
        context : ConversationContext tracking questions and answers.

    Returns:
        Question string, ready to speak to the patient.
    """
    chat = model.start_chat(history=[
        {"role": "user",  "parts": [SYSTEM_PROMPT]},
        {"role": "model", "parts": ["Understood. One question only."]},
    ])
    response = chat.send_message(build_prompt(state, context))
    question = re.sub(r"^(question|q)[:\-]?\s*", "", response.text.strip(), flags=re.IGNORECASE)
    if question and not question.endswith("?"):
        question += "?"
    return question

---
## Usage

### Option A — Single question selection

In [ ]:
state   = {"chief_complaint": "headache", "body_part": "head"}
context = ConversationContext()

question = select_next_question(state, context)
print(question)

### Option B — Simulated conversation loop

In [ ]:
state   = {"chief_complaint": "chest pain", "body_part": "chest", "red_flags_present": True}
context = ConversationContext()

# Simulated patient answers — replace with Model A transcriptions in production
simulated_answers = [
    "It's very bad, maybe eight out of ten.",
    "Yes, it goes to my left arm.",
    "I'm having trouble breathing deeply.",
]

for i, answer in enumerate(simulated_answers):
    q = select_next_question(state, context)
    context.add_turn(q, answer)
    print(f"Q{i+1}: {q}")
    print(f"A{i+1}: {answer}\n")